In [ ]:
# Phase 1 single-cell pipeline: Kaggle download -> preprocessing -> multi-model training -> evaluation -> selection -> second opinion -> Grad-CAM -> save outputs
import json
import os
import random
import shutil
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras import Model
from tensorflow.keras.applications import DenseNet121, InceptionV3, ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('TensorFlow version:', tf.__version__)

try:
    from google.colab import files, drive
except ImportError as exc:
    raise RuntimeError('This notebook is designed to run in Google Colab.') from exc

drive.mount('/content/drive')

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 12

OUTPUT_DIR = Path('/content/pneumonia_phase1_outputs')
DOWNLOAD_DIR = Path('/content/dataset')
data_dir = '/content/dataset/chest_xray'
DATASET_NAME = 'paultimothymooney/chest-xray-pneumonia'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style('whitegrid')

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

def upload_kaggle_json() -> Path:
    kaggle_dir = Path('/root/.kaggle')
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / 'kaggle.json'
    if kaggle_json.exists():
        os.chmod(kaggle_json, 0o600)
        print('kaggle.json already exists at:', kaggle_json)
        return kaggle_json
    print('Upload kaggle.json now...')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('kaggle.json was not uploaded.')
    shutil.move('kaggle.json', kaggle_json)
    os.chmod(kaggle_json, 0o600)
    print('kaggle.json uploaded and configured at:', kaggle_json)
    return kaggle_json

print('=== Step 1: Kaggle setup and download ===')
get_ipython().system('pip -q install kaggle')
_ = upload_kaggle_json()

if not Path(data_dir).exists():
    print('Downloading dataset from Kaggle...')
    get_ipython().system(f'kaggle datasets download -d {DATASET_NAME} -p /content/dataset --unzip')

if not Path(data_dir).exists():
    raise FileNotFoundError(f'Expected dataset directory not found: {data_dir}')

train_dir = os.path.join(data_dir, 'train')
test_dir = os.path.join(data_dir, 'test')
required_classes = ['NORMAL', 'PNEUMONIA']

print('=== Step 2: Validate dataset structure (train/test only) ===')
print('data_dir :', data_dir, 'exists ->', os.path.isdir(data_dir))
print('train_dir:', train_dir, 'exists ->', os.path.isdir(train_dir))
print('test_dir :', test_dir, 'exists ->', os.path.isdir(test_dir))

for split_name, split_root in [('train', train_dir), ('test', test_dir)]:
    if not os.path.isdir(split_root):
        raise FileNotFoundError(f'Missing split folder: {split_root}')
    print(f'\n{split_name}/')
    for class_name in required_classes:
        class_dir = os.path.join(split_root, class_name)
        if not os.path.isdir(class_dir):
            raise FileNotFoundError(f'Missing class folder: {class_dir}')
        count = len([f for f in os.listdir(class_dir) if not f.startswith('.')])
        print(f'  - {class_name}: {count}')

class_names = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
with open(OUTPUT_DIR / 'class_names.json', 'w') as f:
    json.dump(class_names, f)
print('Class names:', class_names)
print('Saved class_names.json to', OUTPUT_DIR / 'class_names.json')

print('=== Step 3: Data generators with validation split from train ===')
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2,
)
test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=IMAGE_SIZE, class_mode='binary',
    batch_size=BATCH_SIZE, subset='training', shuffle=True, seed=SEED
)
validation_generator = train_datagen.flow_from_directory(
    train_dir, target_size=IMAGE_SIZE, class_mode='binary',
    batch_size=BATCH_SIZE, subset='validation', shuffle=False, seed=SEED
)
test_generator = test_datagen.flow_from_directory(
    test_dir, target_size=IMAGE_SIZE, class_mode='binary',
    batch_size=BATCH_SIZE, shuffle=False
)

print('Training samples  :', train_generator.samples)
print('Validation samples:', validation_generator.samples)
print('Test samples      :', test_generator.samples)
print('Class indices     :', train_generator.class_indices)

train_counts = Counter(train_generator.classes.tolist())
class_weight = {
    0: train_generator.samples / (2.0 * max(train_counts.get(0, 1), 1)),
    1: train_generator.samples / (2.0 * max(train_counts.get(1, 1), 1)),
}
print('class_weight:', class_weight)

print('=== Step 4: Visualization fix (dynamic path) ===')
normal_dir = os.path.join(data_dir, 'train', 'NORMAL')
normal_images = sorted([f for f in os.listdir(normal_dir) if not f.startswith('.')])
if len(normal_images) < 9:
    raise RuntimeError(f'Expected at least 9 images in {normal_dir}, found {len(normal_images)}')

plt.figure(figsize=(12, 12))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    img_path = os.path.join(normal_dir, normal_images[i])
    img = plt.imread(img_path)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
plt.suptitle('Sample NORMAL X-rays', fontsize=14)
plt.tight_layout()
plt.show()

def threshold_predictions(probabilities: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    return (probabilities >= threshold).astype(int)

def confidence_band(probability: float) -> tuple[str, float]:
    confidence = float(max(probability, 1.0 - probability))
    if confidence > 0.85:
        return 'high', confidence
    if confidence >= 0.60:
        return 'medium', confidence
    return 'low', confidence

def load_image_for_inference(image_path: str) -> np.ndarray:
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMAGE_SIZE)
    arr = tf.keras.preprocessing.image.img_to_array(img)
    return arr

print('=== Step 5: Model factory ===')
INPUT_SHAPE = IMAGE_SIZE + (3,)
MODEL_SPECS = {
    'ResNet50': {'backbone': ResNet50},
    'DenseNet121': {'backbone': DenseNet121},
    'InceptionV3': {'backbone': InceptionV3},
}

def build_transfer_model(model_name: str) -> tuple[Model, tf.keras.Model]:
    base_model = MODEL_SPECS[model_name]['backbone'](
        include_top=False,
        weights='imagenet',
        input_shape=INPUT_SHAPE,
    )
    base_model.trainable = False

    inputs = Input(shape=INPUT_SHAPE, name=f'{model_name}_input')
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D(name=f'{model_name}_gap')(x)
    x = BatchNormalization(name=f'{model_name}_bn')(x)
    x = Dense(256, activation='relu', name=f'{model_name}_dense_1')(x)
    x = Dropout(0.35, name=f'{model_name}_dropout')(x)
    outputs = Dense(1, activation='sigmoid', name=f'{model_name}_output')(x)

    model = Model(inputs, outputs, name=model_name)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model, base_model

callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6, verbose=1),
]

trained_models = {}

def evaluate_predictions(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    y_pred = threshold_predictions(y_prob, threshold=threshold)
    cm = confusion_matrix(y_true, y_pred)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = 0
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'confusion_matrix': cm,
        'classification_report': classification_report(y_true, y_pred, target_names=class_names, zero_division=0),
    }

def run_experiment(model_name: str) -> dict:
    checkpoint_path = f'/content/drive/MyDrive/checkpoints/{model_name}.keras'

    train_generator.reset()
    validation_generator.reset()
    test_generator.reset()

    if os.path.exists(checkpoint_path):
        print(f'✅ Loading existing model for {model_name} (skipping training)')
        model = tf.keras.models.load_model(checkpoint_path)
        base_model = None
        history = {'skipped': True}
    else:
        print(f'🚀 Training new model: {model_name}')
        model, base_model = build_transfer_model(model_name)

        checkpoint_dir = Path('/content/drive/MyDrive/checkpoints')
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_cb = ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_loss',
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        )
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6, verbose=1),
            checkpoint_cb
        ]

        history_obj = model.fit(
            train_generator,
            validation_data=validation_generator,
            epochs=EPOCHS,
            class_weight=class_weight,
            callbacks=callbacks,
            verbose=1,
        )
        history = history_obj.history

    test_generator.reset()
    y_prob = model.predict(test_generator, verbose=0).ravel()
    y_true = test_generator.classes.copy()
    metrics = evaluate_predictions(y_true, y_prob)

    trained_models[model_name] = {
        'model': model,
        'base_model': base_model,
        'history': history,
        'metrics': metrics,
        'y_true': y_true,
        'y_prob': y_prob,
    }
    print(metrics['classification_report'])
    print(f'{model_name} F1-score: {metrics["f1_score"]:.4f}')
    return trained_models[model_name]

print('=== Step 6: Train models ===')
for model_name in ['ResNet50', 'DenseNet121', 'InceptionV3']:
    run_experiment(model_name)

print('=== Step 7: metrics_df and model comparison ===')
metrics_df = pd.DataFrame([
    {
        'model_name': name,
        'accuracy': info['metrics']['accuracy'],
        'precision': info['metrics']['precision'],
        'recall': info['metrics']['recall'],
        'f1_score': info['metrics']['f1_score'],
        'tn': info['metrics']['tn'],
        'fp': info['metrics']['fp'],
        'fn': info['metrics']['fn'],
        'tp': info['metrics']['tp'],
    }
    for name, info in trained_models.items()
]).sort_values('f1_score', ascending=False).reset_index(drop=True)
print(metrics_df[['model_name', 'accuracy', 'precision', 'recall', 'f1_score']])
metrics_df.to_csv(OUTPUT_DIR / 'metrics_df.csv', index=False)

plt.figure(figsize=(12, 5))
melted = metrics_df[['model_name', 'accuracy', 'precision', 'recall', 'f1_score']].melt(
    id_vars='model_name', var_name='metric', value_name='score'
)
sns.barplot(data=melted, x='model_name', y='score', hue='metric')
plt.ylim(0, 1)
plt.title('Model Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print('=== Step 8: Select top-2 models and save ===')
selected = metrics_df.head(2)['model_name'].tolist()
if len(selected) < 2:
    raise RuntimeError('Need at least two trained models.')
model1_name, model2_name = selected
model1 = trained_models[model1_name]['model']
model2 = trained_models[model2_name]['model']
model1_path = OUTPUT_DIR / 'best_model_1.keras'
model2_path = OUTPUT_DIR / 'best_model_2.keras'
model1.save(model1_path)
model2.save(model2_path)
print('Top models:', model1_name, model2_name)
print('Saved:', model1_path)
print('Saved:', model2_path)

def predict_single_image(model_name: str, image_array: np.ndarray) -> dict:
    model = trained_models[model_name]['model']
    image_batch = np.expand_dims(image_array.astype('float32') / 255.0, axis=0)
    probability = float(model.predict(image_batch, verbose=0).ravel()[0])
    pred_idx = int(probability >= 0.5)
    conf_label, conf_score = confidence_band(probability)
    return {
        'model_name': model_name,
        'probability': probability,
        'prediction_index': pred_idx,
        'prediction_label': class_names[pred_idx],
        'confidence_label': conf_label,
        'confidence_score': conf_score,
    }

def run_second_opinion(image_array: np.ndarray) -> dict:
    prediction_model1 = predict_single_image(model1_name, image_array)
    prediction_model2 = predict_single_image(model2_name, image_array)
    agreement_flag = prediction_model1['prediction_index'] == prediction_model2['prediction_index']
    low_confidence_flag = (
        prediction_model1['confidence_label'] == 'low'
        or prediction_model2['confidence_label'] == 'low'
    )
    warning_flag = (not agreement_flag) or low_confidence_flag
    return {
        'prediction_model1': prediction_model1,
        'prediction_model2': prediction_model2,
        'agreement_flag': agreement_flag,
        'low_confidence_flag': low_confidence_flag,
        'warning_flag': warning_flag,
    }

print('=== Step 9: Second opinion demo ===')
test_generator.reset()
sample_path = test_generator.filepaths[0]
sample_label = int(test_generator.classes[0])
sample_image = load_image_for_inference(sample_path)
second_opinion_result = run_second_opinion(sample_image)
print('Sample path:', sample_path)
print('Ground truth:', class_names[sample_label])
print('prediction_model1:', second_opinion_result['prediction_model1'])
print('prediction_model2:', second_opinion_result['prediction_model2'])
print('agreement_flag:', second_opinion_result['agreement_flag'])
print('warning_flag:', second_opinion_result['warning_flag'])

print('=== Step 10: Grad-CAM helpers and samples ===')
def find_last_conv_layer_name(model: tf.keras.Model) -> str:
    for layer in reversed(model.layers):
        output_shape = getattr(layer, 'output_shape', None)
        if output_shape is not None and len(output_shape) == 4:
            return layer.name
    raise ValueError('Could not find convolutional layer for Grad-CAM.')

def make_gradcam_heatmap(image_array, model):
    import tensorflow as tf
    import numpy as np

    img = np.expand_dims(image_array.astype("float32") / 255.0, axis=0)
    img = tf.convert_to_tensor(img)

    # Get last conv layer safely
    last_conv_layer = None
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv_layer = layer.name
            break

    if last_conv_layer is None:
        raise ValueError("No Conv2D layer found in model.")

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img)

        # 🔥 FIX: use predicted class dynamically
        pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]

    grads = tape.gradient(loss, conv_outputs)

    # Avoid None gradients
    if grads is None:
        return np.zeros((IMAGE_SIZE[0], IMAGE_SIZE[1]))

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(pooled_grads * conv_outputs, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()

def save_gradcam_overlay(image_array: np.ndarray, heatmap: np.ndarray, output_path: Path, title: str) -> None:
    plt.figure(figsize=(7, 7))
    image_display = np.clip(image_array / (np.max(image_array) + 1e-8), 0, 1)
    plt.imshow(image_display)
    plt.imshow(heatmap, cmap='jet', alpha=0.35)
    plt.axis('off')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.05)
    plt.show()
    plt.close()

gradcam_dir = OUTPUT_DIR / 'gradcam_samples'
gradcam_dir.mkdir(parents=True, exist_ok=True)
for idx, model_name in enumerate([model1_name, model2_name], start=1):
    m = trained_models[model_name]['model']
    p = float(m.predict(np.expand_dims(sample_image.astype('float32') / 255.0, axis=0), verbose=0).ravel()[0])
    heatmap = make_gradcam_heatmap(sample_image, m)
    out_path = gradcam_dir / f'gradcam_{idx}_{model_name}.png'
    save_gradcam_overlay(sample_image, heatmap, out_path, f'{model_name} | prob={p:.3f}')
    print('Saved Grad-CAM:', out_path)

print('=== Step 11: Training/Validation curves + confusion matrices ===')
plt.figure(figsize=(16, 5))
for i, metric_name in enumerate(['loss', 'accuracy', 'precision'], start=1):
    plt.subplot(1, 3, i)
    for name, info in trained_models.items():
        hist = info['history']
        if metric_name in hist and f'val_{metric_name}' in hist:
            plt.plot(hist[metric_name], label=f'{name} train')
            plt.plot(hist[f'val_{metric_name}'], linestyle='--', label=f'{name} val')
    plt.title(metric_name.title())
    plt.xlabel('Epoch')
    plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(trained_models), figsize=(6 * len(trained_models), 5))
if len(trained_models) == 1:
    axes = [axes]
for ax, (name, info) in zip(axes, trained_models.items()):
    cm = info['metrics']['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names, rotation=0)
plt.tight_layout()
plt.show()

summary_payload = {
    'data_dir': data_dir,
    'class_names': class_names,
    'top_models': [model1_name, model2_name],
    'metrics_file': str(OUTPUT_DIR / 'metrics_df.csv'),
    'model1_path': str(model1_path),
    'model2_path': str(model2_path),
    'gradcam_dir': str(gradcam_dir),
}
with open(OUTPUT_DIR / 'phase1_summary.json', 'w') as f:
    json.dump(summary_payload, f, indent=2)

print('=== Completed ===')
print('Artifacts in:', OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print('-', p)